# 01 — Tweet Collection
Scrapes recent tweets (last 30 days) for each Premier League Big 6 team using `twscrape`.

**Before running:** fill in your X credentials in the cell below.

**Run once** — output CSVs are saved to `data/raw/` and reused in notebook 02.

In [ ]:
# Install dependencies if needed
# !pip install twscrape pandas

In [ ]:
import asyncio
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path
import twscrape

In [ ]:
# ── CREDENTIALS ───────────────────────────────────────────
X_USERNAME = "KippInsights"
X_PASSWORD = "your_password"
X_EMAIL    = "kipp.batchelor@outlook.com"
# Grab from browser DevTools → Application → Cookies → x.com
AUTH_TOKEN = "your_auth_token_here"
CT0        = "your_ct0_here"
# ──────────────────────────────────────────────────────────

RAW_DIR         = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

TWEETS_PER_TEAM = 500   # raise if you want richer data

SINCE = (datetime.now(timezone.utc) - timedelta(days=30)).strftime("%Y-%m-%d")
UNTIL =  datetime.now(timezone.utc).strftime("%Y-%m-%d")

print(f"Date window: {SINCE}  →  {UNTIL}")

In [ ]:
# Search queries per team
# -is:retweet keeps original tweets only; lang:en keeps English
TEAMS = {
    "Arsenal":    "(Arsenal OR #AFC OR #COYG) lang:en -is:retweet",
    "Chelsea":    "(Chelsea OR #CFC OR #KTBFFH) lang:en -is:retweet",
    "Liverpool":  "(Liverpool OR #LFC OR #YNWA) lang:en -is:retweet",
    "Man City":   "(\"Man City\" OR \"Manchester City\" OR #MCFC) lang:en -is:retweet",
    "Man United": "(\"Man United\" OR \"Manchester United\" OR #MUFC) lang:en -is:retweet",
    "Tottenham":  "(Tottenham OR Spurs OR #THFC OR #COYS) lang:en -is:retweet",
}

In [ ]:
from twscrape.utils import parse_cookies

async def collect_all():
    api = twscrape.API()

    # ── Debug: verify cookies parse correctly ──────────────
    cookies = f"auth_token={AUTH_TOKEN}; ct0={CT0}"
    parsed  = parse_cookies(cookies)
    print(f"auth_token present: {'auth_token' in parsed} (len={len(parsed.get('auth_token',''))})")
    print(f"ct0        present: {'ct0' in parsed} (len={len(parsed.get('ct0',''))})")
    if 'ct0' not in parsed or len(parsed.get('ct0','')) < 10:
        raise ValueError("ct0 cookie missing or too short — re-copy it from DevTools")
    # ────────────────────────────────────────────────────────

    # Remove any stale account entry
    try:
        await api.pool.delete_accounts(X_USERNAME)
        print(f"Removed stale account entry for {X_USERNAME}")
    except Exception as e:
        print(f"delete_accounts: {e}")

    # Add account with cookies — if ct0 is present twscrape marks it active=True
    await api.pool.add_account(
        X_USERNAME, X_PASSWORD, X_EMAIL, X_PASSWORD,
        cookies=cookies,
    )

    # Confirm active status
    accounts = await api.pool.get_all()
    for acc in accounts:
        print(f"Account '{acc.username}': active={acc.active}, cookies={list(acc.cookies.keys())}")
        if not acc.active:
            raise ValueError("Account not active after adding cookies — ct0 may be invalid or expired")

    all_frames = []

    for team, query in TEAMS.items():
        full_query = f"{query} since:{SINCE} until:{UNTIL}"
        print(f"\nScraping: {team}")

        records = []
        async for tweet in api.search(full_query, limit=TWEETS_PER_TEAM):
            records.append({
                "id":       tweet.id,
                "team":     team,
                "date":     tweet.date,
                "text":     tweet.rawContent,
                "likes":    tweet.likeCount,
                "retweets": tweet.retweetCount,
                "replies":  tweet.replyCount,
            })

        df = pd.DataFrame(records)
        out = RAW_DIR / f"{team.lower().replace(' ', '_')}.csv"
        df.to_csv(out, index=False)
        print(f"  Saved {len(df)} tweets → {out}")
        all_frames.append(df)

    combined = pd.concat(all_frames, ignore_index=True)
    combined.to_csv(RAW_DIR / "all_teams.csv", index=False)
    print(f"\n✓ Done — {len(combined)} tweets total saved to data/raw/all_teams.csv")
    return combined


df_all = await collect_all()

In [ ]:
# Sanity check
print(df_all.groupby("team")["id"].count().rename("tweet_count").to_string())
df_all.head(3)